In [1]:
import pandas as pd
import sqlite3
import plotly.graph_objects as go
import numpy as np

connection=sqlite3.connect('../data/checking-logs.sqlite')

In [10]:
query = '''
SELECT uid, timestamp, numTrials FROM checker
WHERE uid LIKE 'user_%' AND status="ready" AND labname="project1"
'''

commits = pd.read_sql(sql=query, con=connection, parse_dates=['timestamp'])
commits['date'] = commits['timestamp'].dt.floor('d')

commits = commits.groupby(['date', 'uid']).max()['numTrials'].unstack()
commits = commits.ffill().fillna(0)

In [9]:
days = np.arange(1, len(commits.index) + 1)

initial_data = [go.Scatter(x=[], y=[], mode='lines+markers', name=uid, line=dict(width=2)) for uid in commits.columns]

frames = [
    go.Frame(
        data=[go.Scatter(x=days[:i+1], y=commits.iloc[:i+1, j], mode='lines+markers', name=commits.columns[j]) for j in range(len(commits.columns))],
        name=f"Day {i+1}"
    ) for i in range(len(commits))
]

fig = go.Figure(
    data=initial_data,
    frames=frames,
    layout=go.Layout(
        title='Dynamic of commits per user in project1',
        xaxis=dict(range=[1, len(commits)+1], dtick=2),
        yaxis=dict(range=[0, commits.max().max() + 2]),
        hovermode='x unified',
        updatemenus=[dict(type='buttons', buttons=[dict(label='Play', method='animate', args=[None, dict(frame=dict(duration=500, redraw=True), fromcurrent=True)])])],
        legend=dict(x=1.05, y=0.5)
    )
)


fig.update_layout(width=1200, height=700, plot_bgcolor='rgba(240,240,240,0.8)', margin=dict(r=150))
fig.show()

In [11]:
connection.close()